In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

output_notebook()
hv.extension('bokeh')


Loading BokehJS ...

In [5]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 

# Load the pickle file
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
pickle_file = save_path / f'msn_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)
print(f"Unified DataFrame loaded from: {pickle_file}")
print(f"DataFrame shape: {cell_df.shape}")
print(cell_df.info())
cell_df.head()

Unified DataFrame loaded from: /home/barak/Projects/population-analysis/data/unified_cell_trial_data/msn_fiona_cell_trial_data.pkl
DataFrame shape: (1906402, 27)
<class 'pandas.core.frame.DataFrame'>
Index: 1906402 entries, 0 to 2428536
Data columns (total 27 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   cell_ID                 int64  
 1   cell_type               object 
 2   maestro_ID              int64  
 3   problem                 object 
 4   grade                   int64  
 5   filename                object 
 6   trial_name              object 
 7   reaction_time           float64
 8   go_cue                  int64  
 9   stop_cue                float64
 10  trial_failed            bool   
 11  ssd_len                 int64  
 12  ssd_number              float64
 13  type                    object 
 14  first_relevant_saccade  object 
 15  segs_durations          object 
 16  segs_times              object 
 17  trial_length          

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9868,msn,3,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[],fi210824,a,614,fi210824a
1,9869,msn,4,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[2197.27],fi210824,a,614,fi210824a
2,9872,msn,7,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[925.18, 985.9, 994.25, 2055.17]",fi210824,a,614,fi210824a
3,9873,msn,8,NaN,7,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[904.25, 1356.03, 1693.2, 1910.72]",fi210824,a,614,fi210824a
4,9876,msn,11,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[1076.68],fi210824,a,614,fi210824a


In [6]:
print(cell_df[['cell_type', 'cell_ID']].drop_duplicates('cell_ID')['cell_type'].value_counts())

cell_df['grade'].value_counts()


cell_type
pu msn    1773
msn       1254
Name: count, dtype: int64


grade
8    1454797
7     404157
6      47448
Name: count, dtype: int64

In [7]:
df = cell_df[cell_df['neural_data'].apply(lambda x: len(x) == 0)]
# df = df[df['grade'] == 8]
df = df[df['session'] == 'fi211109']
df = df[df['trial_number'] == 270]
df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
1464415,2013,pu msn,30,NaN,7,fi211109a.0270,GO_L,157.0,977,NaN,...,2128,0.0,"[[150, 225], [834, 895], [1134, 1211], [1704, ...",None,180,[],fi211109,a,270,fi211109a
1464426,2032,pu msn,49,NaN,8,fi211109a.0270,GO_L,157.0,977,NaN,...,2128,0.0,"[[150, 225], [834, 895], [1134, 1211], [1704, ...",None,180,[],fi211109,a,270,fi211109a


In [8]:
cell_df.columns

Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')

In [9]:
cell_df = cell_df[cell_df['grade'] <= 8].copy().reset_index(drop=True)
cell_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9868,msn,3,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[],fi210824,a,614,fi210824a
1,9869,msn,4,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[2197.27],fi210824,a,614,fi210824a
2,9872,msn,7,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[925.18, 985.9, 994.25, 2055.17]",fi210824,a,614,fi210824a
3,9873,msn,8,NaN,7,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[904.25, 1356.03, 1693.2, 1910.72]",fi210824,a,614,fi210824a
4,9876,msn,11,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[1076.68],fi210824,a,614,fi210824a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1906397,2930,pu msn,99,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[78.76, 246.83, 746.31, 892.11, 941.41, 1118.6...",fi211020,a,451,fi211020a
1906398,2931,pu msn,100,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[199.58, 234.53, 289.33, 297.48, 591.91, 648.4...",fi211020,a,451,fi211020a
1906399,2933,pu msn,102,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[84.61, 316.63, 366.98, 522.18, 640.41, 656.28...",fi211020,a,451,fi211020a
1906400,2934,pu msn,103,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[66.76, 141.41, 165.56, 270.36, 482.13, 503.13...",fi211020,a,451,fi211020a


In [10]:
cell_df['session'].nunique()


61

In [11]:
## Cell 3: Trial Type Distribution and Success Rates

# Create summary statistics for plotting
trial_summary = cell_df.groupby(['type', 'trial_failed']).size().reset_index(name='count')
trial_summary['outcome'] = trial_summary['trial_failed'].map({False: 'Success', True: 'Failed'})

# Calculate success rates by trial type
success_rates = cell_df.groupby('type').agg({
    'trial_failed': ['count', 'sum', 'mean']
}).round(3)
success_rates.columns = ['total_trials', 'failed_trials', 'failure_rate']
success_rates['success_rate'] = (1 - success_rates['failure_rate']) * 100
success_rates['failure_rate'] *= 100
print("Success rates by trial type:")
print(success_rates)

# Create the main visualization
plot1 = trial_summary.hvplot.bar(
    x='type', y='count', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Trial Distribution by Type and Outcome',
    xlabel='Trial Type',
    ylabel='Number of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],  # Green for success, red for failed
    legend='top_right'
)

plot1.opts(
    fontsize={'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12},
)
# success_rates

Success rates by trial type:
      total_trials  failed_trials  failure_rate  success_rate
type                                                         
CONT        443414          64493          14.5          85.5
GO         1054323          36610           3.5          96.5
STOP        408665         186428          45.6          54.4


:Bars   [type,outcome]   (count)

In [12]:
## Cell 4: Success Rates by Trial Type (Percentage View)
# Create percentage view of success rates
trial_pct = cell_df.groupby('type').apply(
    lambda x: pd.Series({
        'Success': (1 - x['trial_failed'].mean()) * 100,
        'Failed': x['trial_failed'].mean() * 100
    })
).reset_index()

trial_pct_melted = trial_pct.melt(id_vars='type', var_name='outcome', value_name='percentage')

plot2 = trial_pct_melted.hvplot.bar(
    x='type', y='percentage', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Success Rate by Trial Type (%)',
    xlabel='Trial Type',
    ylabel='Percentage of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],
    legend='top',
    ylim=(0, 100)
)

plot2
# trial_pct
# trial_pct_melted

/tmp/ipykernel_225518/925790005.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trial_pct = cell_df.groupby('type').apply(


:Bars   [type,outcome]   (percentage)

In [13]:
from bokeh.palettes import Colorblind

# Histogram of cells per grade
print("=== CELLS PER GRADE ANALYSIS ===")

# Get the grade distribution for all cells
grade_counts = cell_df['grade'].value_counts().sort_index()
print(f"Grade distribution:")
for grade, count in grade_counts.items():
    print(f"  Grade {grade}: {count:,} cells")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Grade range: {cell_df['grade'].min()} - {cell_df['grade'].max()}")
print(f"Mean grade: {cell_df['grade'].mean():.2f}")
print(f"Median grade: {cell_df['grade'].median():.1f}")

# Create bar plot using hvplot with different colors per bar
grade_counts_df = cell_df.groupby('grade').size().reset_index(name='count')

# Create individual bars with different colors
bars = []
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
n_bars = len(grade_counts_df)
palette_size = min(8, max(3, n_bars))
colors = Colorblind[palette_size]

for i, (grade, count) in enumerate(zip(grade_counts_df['grade'], grade_counts_df['count'])):
    bar = hv.Bars([(grade, count)], kdims='grade', vdims='count').opts(
        color=colors[i], 
        alpha=0.8,
        width=10,
    )
    bars.append(bar)

# Overlay all bars
grade_bar = hv.Overlay(bars).opts(
    title=f'{monkey.title()} - Distribution of Cells per Grade',
    xlabel='Grade',
    ylabel='Number of Cells',
    width=700, height=400,
    fontsize=font_dict,
    xticks=list(grade_counts_df['grade'])
)

grade_bar

=== CELLS PER GRADE ANALYSIS ===
Grade distribution:
  Grade 6: 47,448 cells
  Grade 7: 404,157 cells
  Grade 8: 1,454,797 cells

Total cells: 1,906,402
Grade range: 6 - 8
Mean grade: 7.74
Median grade: 8.0


:Overlay
   .Bars.I   :Bars   [grade]   (count)
   .Bars.II  :Bars   [grade]   (count)
   .Bars.III :Bars   [grade]   (count)

In [14]:
# Violin plot of cell distribution by trial type and outcome
print("=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===")

# Count cells per trial for all trials - using correct column names
cells_per_trial = cell_df.groupby(['filename', 'type', 'trial_failed']).size().reset_index(name='cell_count')

# # Add outcome labels
cells_per_trial['outcome'] = cells_per_trial['trial_failed'].map({False: 'Success', True: 'Failed'})
cells_per_trial
print(f"Total trials analyzed: {len(cells_per_trial):,}")

print(f"\nCells per trial statistics by type and outcome:")
for trial_type in cells_per_trial['type'].unique():
    for outcome in ['Success', 'Failed']:
        type_outcome_data = cells_per_trial[
            (cells_per_trial['type'] == trial_type) & 
            (cells_per_trial['outcome'] == outcome)
        ]['cell_count']
        if len(type_outcome_data) > 0:
            print(f"  {trial_type} {outcome}: Mean={type_outcome_data.mean():.1f}, "
                  f"Median={type_outcome_data.median():.1f}, Min={type_outcome_data.min()}, "
                  f"Max={type_outcome_data.max()}, Trials={len(type_outcome_data)}")

# Create rotated violin plot with split by outcome using hv.Violin
violin_plot = hv.Violin(
    cells_per_trial, kdims=['type', 'outcome'], vdims='cell_count'
).opts(
    opts.Violin(
        show_legend=True, height=500, width=800,
        violin_color=hv.dim('outcome').str(),
        legend_position='top_right',
        split='outcome',
        title=f'{monkey.title()} - Distribution of Cells per Trial by Type and Outcome',
        xlabel='Trial Type',
        ylabel='Number of Cells per Trial',
        show_grid=True,
        violin_width=2,
        invert_axes=True,  # Keep normal orientation (vertical violins)
        tools=['hover'],
        fontsize=font_dict
    )
)


violin_plot

=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===
Total trials analyzed: 80,789

Cells per trial statistics by type and outcome:
  CONT Success: Mean=23.9, Median=19.0, Min=1, Max=73, Trials=15834
  CONT Failed: Mean=22.8, Median=18.0, Min=1, Max=73, Trials=2826
  GO Success: Mean=23.7, Median=19.0, Min=1, Max=73, Trials=43002
  GO Failed: Mean=21.0, Median=17.0, Min=1, Max=68, Trials=1740
  STOP Success: Mean=22.7, Median=18.0, Min=1, Max=73, Trials=9783
  STOP Failed: Mean=24.5, Median=20.0, Min=1, Max=73, Trials=7604


:Violin   [type,outcome]   (cell_count)

In [15]:
cell_df['ssd_len'].describe()

count    1.906402e+06
mean     3.136417e+02
std      1.637894e+02
min      2.400000e+01
25%      1.680000e+02
50%      4.500000e+02
75%      4.500000e+02
max      5.500000e+02
Name: ssd_len, dtype: float64

In [16]:
# Bar plot of cell count by cell type and trial type
print("=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===")

# Get the cell type distribution by trial type
cell_type_trial_counts = cell_df.groupby(['type']).size().reset_index(name='count')

# Calculate percentages within each trial type
trial_totals = cell_df.groupby('type').size()
cell_type_trial_counts['percentage'] = cell_type_trial_counts.apply(
    lambda row: (row['count'] / trial_totals[row['type']]) * 100, axis=1
)

# print(f"Cell type distribution by trial type:")
# for trial_type in cell_df['type'].unique():
    # print(f"\n{trial_type} trials:")
    # trial_data = cell_type_trial_counts[cell_type_trial_counts['type'] == trial_type].sort_values('count', ascending=False)
    # for _, row in trial_data.iterrows():
    #     print(f"  {row['type']}: {row['count']:,} cells ({row['percentage']:.1f}%)")

# print(f"\nTotal cells: {len(cell_df):,}")
# print(f"Trial types: {cell_df['type'].unique()}")

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='type', y='count', 
    title=f'{monkey.title()} - Distribution of Cells by Cell Type and Trial Type',
    xlabel='Trial Type',
    ylabel='Number of Cells',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    colormap=('#2E8B57', '#FF8C00', '#4169E1'),  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)

# Apply additional styling options
cell_type_bar = cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'},
)

cell_type_bar

=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===


:Bars   [type]   (count)